# LongFact + LongFact++ → Hindi (Sarvam-Translate)

Translates every prompt of **LongFact** (Wei et al., 2024; 2,280 prompts, from `google-deepmind/long-form-factuality`) and **LongFact++** (Obeso et al., 2025; `obalcells/longfact-augmented-prompts`) into Hindi and saves them to a **public** Hugging Face dataset `<your-username>/longfact-hindi`, in the **same format as the originals**:

| Source | Output path in the repo | Format |
|---|---|---|
| LongFact-Objects | `longfact/longfact-objects_gpt4_01-12-2024_noduplicates/longfact-objects_<topic>.jsonl` | one JSON per line: `prompt` (Hindi), `canary` (unchanged) — same files, same line order |
| LongFact-Concepts | `longfact/longfact-concepts_gpt4_01-10-2024_noduplicates/longfact-concepts_<topic>.jsonl` | same as above |
| LongFact++ | `longfact-augmented-prompts/train.jsonl` | columns `question` (Hindi), `topic`, `difficulty` (unchanged) — same row order |

Line *i* of each output file is the translation of line *i* of the original, so the English is always recoverable.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Add a Colab secret named **`HF_TOKEN`** (write access) and enable notebook access. Then Runtime → Run all.
If Colab disconnects, just Run all again: progress is checkpointed to the repo (`checkpoint/progress.jsonl`) and resumes.

In [ ]:
!pip install -q -U transformers datasets accelerate

In [ ]:
import json, os, time, urllib.request
import torch
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "sarvamai/sarvam-translate"   # Gemma-3-4B fine-tune for 22 Indian languages (Sarvam AI + AI4Bharat)
TARGET_LANG = "Hindi"
REPO_NAME = "longfact-hindi"
BATCH_SIZE = 16          # halves automatically on GPU out-of-memory
MAX_NEW_TOKENS = 384
PUSH_EVERY = 200         # upload the checkpoint every N new translations
OUT_DIR = "longfact-hindi"
PROGRESS = "progress.jsonl"

def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)

HF_TOKEN = userdata.get("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
repo_id = f"{api.whoami()['name']}/{REPO_NAME}"
api.create_repo(repo_id, repo_type="dataset", private=False, exist_ok=True)
try:   # make it public in case an earlier run created it private
    api.update_repo_settings(repo_id, repo_type="dataset", private=False)
except AttributeError:   # older huggingface_hub
    api.update_repo_visibility(repo_id, private=False, repo_type="dataset")
log(f"Output repo (public): https://huggingface.co/datasets/{repo_id}")

In [ ]:
# ---------------- load the original prompts, keeping their files and order ----------------
GH = "https://raw.githubusercontent.com/google-deepmind/long-form-factuality/main/longfact"
LF_FOLDERS = {"longfact-objects": "longfact-objects_gpt4_01-12-2024_noduplicates",
              "longfact-concepts": "longfact-concepts_gpt4_01-10-2024_noduplicates"}
TOPICS = ["20th-century-events", "US-foreign-policy", "accounting", "architecture", "astronomy", "biology",
          "business-ethics", "celebrities", "chemistry", "clinical-knowledge", "computer-science",
          "computer-security", "economics", "electrical-engineering", "gaming", "geography", "global-facts",
          "history", "immigration-law", "international-law", "jurisprudence", "machine-learning", "management",
          "marketing", "mathematics", "medicine", "moral-disputes", "movies", "music", "philosophy", "physics",
          "prehistory", "psychology", "public-relations", "sociology", "sports", "virology", "world-religions"]

longfact = {}   # output path -> list of original rows {"prompt", "canary"}
for prefix, folder in LF_FOLDERS.items():
    for topic in TOPICS:
        path = f"longfact/{folder}/{prefix}_{topic}.jsonl"
        text = urllib.request.urlopen(f"{GH}/{folder}/{prefix}_{topic}.jsonl").read().decode("utf-8")
        longfact[path] = [json.loads(line) for line in text.splitlines() if line.strip()]
log(f"LongFact: {sum(len(v) for v in longfact.values())} prompts in {len(longfact)} files")

lfpp = load_dataset("obalcells/longfact-augmented-prompts", split="train", token=HF_TOKEN)
log(f"LongFact++: {len(lfpp)} prompts")

# one work item per prompt: id -> English text (LongFact first, then LongFact++)
items = []
for path, rows in longfact.items():
    for i, row in enumerate(rows):
        items.append((f"{path}#{i}", row["prompt"]))
for i, row in enumerate(lfpp):
    items.append((f"longfact-augmented-prompts/train.jsonl#{i}", row["question"]))
log(f"Total: {len(items)} prompts")

In [ ]:
# ---------------- resume from the checkpoint in the repo ----------------
done = {}
if api.file_exists(repo_id, f"checkpoint/{PROGRESS}", repo_type="dataset"):
    p = hf_hub_download(repo_id, f"checkpoint/{PROGRESS}", repo_type="dataset", token=HF_TOKEN, force_download=True)
    with open(p, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            done[r["id"]] = r["hi"]
    log(f"Resuming: {len(done)} already translated")
else:
    log("No checkpoint yet, starting fresh")

with open(PROGRESS, "w", encoding="utf-8") as f:
    for k, v in done.items():
        f.write(json.dumps({"id": k, "hi": v}, ensure_ascii=False) + "\n")

def push_checkpoint(msg):
    api.upload_file(path_or_fileobj=PROGRESS, path_in_repo=f"checkpoint/{PROGRESS}",
                    repo_id=repo_id, repo_type="dataset", commit_message=msg)
    log(f"Checkpoint pushed: {msg}")

todo = [(k, en) for k, en in items if k not in done]
log(f"{len(todo)} prompts left")

In [ ]:
# ---------------- load Sarvam-Translate ----------------
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
log(f"Loading {MODEL_ID} on {torch.cuda.get_device_name(0)} ...")
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16,
                                             device_map="cuda", token=HF_TOKEN).eval()

@torch.no_grad()
def translate(texts):
    chats = [tok.apply_chat_template([{"role": "system", "content": f"Translate the text below to {TARGET_LANG}."},
                                      {"role": "user", "content": t}],
                                     tokenize=False, add_generation_prompt=True) for t in texts]
    enc = tok(chats, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

log("Sanity check -> " + translate(["Can you provide information about the V838 Monocerotis star?"])[0])

In [ ]:
# ---------------- translate, checkpointing as we go ----------------
start, since_push, bs, i = time.time(), 0, BATCH_SIZE, 0
while i < len(todo):
    batch = todo[i:i + bs]
    try:
        outs = translate([en for _, en in batch])
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        bs = max(1, bs // 2)
        log(f"GPU out of memory, batch size -> {bs}")
        continue
    with open(PROGRESS, "a", encoding="utf-8") as f:
        for (k, _), hi in zip(batch, outs):
            done[k] = hi
            f.write(json.dumps({"id": k, "hi": hi}, ensure_ascii=False) + "\n")
            if not any("\u0900" <= ch <= "\u097F" for ch in hi):
                log(f"  warning: no Devanagari in output for {k}")
    i += len(batch)
    since_push += len(batch)
    rate = i / (time.time() - start)
    log(f"{len(done)}/{len(items)} done  ({rate:.2f}/s, ~{(len(todo) - i) / rate / 60:.0f} min left)")
    if since_push >= PUSH_EVERY:
        push_checkpoint(f"{len(done)}/{len(items)} translated")
        since_push = 0

push_checkpoint(f"{len(done)}/{len(items)} translated")

In [ ]:
# ---------------- write the final files in the original formats and upload ----------------
# Writes every dataset that is fully translated; anything unfinished is skipped (run again to finish it).
lf_ready = all(f"{p}#{i}" in done for p, rows in longfact.items() for i in range(len(rows)))
lfpp_ready = all(f"longfact-augmented-prompts/train.jsonl#{i}" in done for i in range(len(lfpp)))

if lf_ready:
    for path, rows in longfact.items():
        os.makedirs(os.path.dirname(os.path.join(OUT_DIR, path)), exist_ok=True)
        with open(os.path.join(OUT_DIR, path), "w", encoding="utf-8") as f:
            for i, row in enumerate(rows):
                f.write(json.dumps({"prompt": done[f"{path}#{i}"], "canary": row["canary"]}, ensure_ascii=False) + "\n")
    log("LongFact files written")
else:
    log("LongFact not finished yet - skipped")

if lfpp_ready:
    os.makedirs(os.path.join(OUT_DIR, "longfact-augmented-prompts"), exist_ok=True)
    with open(os.path.join(OUT_DIR, "longfact-augmented-prompts/train.jsonl"), "w", encoding="utf-8") as f:
        for i, row in enumerate(lfpp):
            f.write(json.dumps({"question": done[f"longfact-augmented-prompts/train.jsonl#{i}"],
                                "topic": row["topic"], "difficulty": row["difficulty"]}, ensure_ascii=False) + "\n")
    log("LongFact++ file written")
else:
    log("LongFact++ not finished yet - skipped")

if lf_ready or lfpp_ready:
    api.upload_folder(folder_path=OUT_DIR, repo_id=repo_id, repo_type="dataset",
                      commit_message="LongFact / LongFact++ Hindi translations (Sarvam-Translate)")
    log(f"Uploaded: https://huggingface.co/datasets/{repo_id}")